# HS4002 Week 6

## OLS with Higher-Order Effects, Interaction Terms, and Logistic Regression

Today we cover:
1. Polynomial terms in OLS (age-squared)
2. Interaction effects
3. The Linear Probability Model (LPM)
4. Logistic regression
5. Marginal effects


In [ ]:
# !pip install pandas numpy plotnine statsmodels marginaleffects stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [ ]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# OLS with Higher-Order Polynomials

Begin by regressing income on age. Save as `ols_model1`.

In [ ]:
ols_model1 = smf.ols('income_cont ~ age', data=df).fit()
ols_model1.summary()

## Adding an age-squared term

In Python formulas, use `I(age**2)` — the `I()` wrapper tells the formula parser to treat the expression literally.

In [ ]:
ols_model2 = smf.ols('income_cont ~ age + I(age**2)', data=df).fit()
ols_model2.summary()

## Making a regression table

As in Week 5, the `stargazer` package collects fitted models into a single regression table. Reading models side by side is much easier than scrolling through two separate `.summary()` blocks.

In [ ]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_model1, ols_model2])
sg.title('Income, Age, and Age-Squared')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2'], [1, 1])
sg.covariate_order(['age', 'I(age ** 2)', 'Intercept'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
	'Intercept':   'Intercept'
})

# Ending the cell on the object itself renders the table in the notebook
sg

## Interaction Effects

Add BA education and gender to the model. Save as `ols_model3`.

In [ ]:
ols_model3 = smf.ols('income_cont ~ age + I(age**2) + ba_binary + sex_woman', data=df).fit()
ols_model3.summary()

Now add an interaction term between BA and gender. In statsmodels formulas, `:` creates an interaction.

Save as `ols_model4`.

In [ ]:
ols_model4 = smf.ols(
	'income_cont ~ age + I(age**2) + ba_binary + sex_woman + ba_binary:sex_woman',
	data=df
).fit()

In [ ]:
sg = Stargazer([ols_model3, ols_model4])
sg.title('Interaction Between BA and Gender')
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 3', 'Model 4'], [1, 1])
sg.covariate_order([
	'age',
	'I(age ** 2)',
	'ba_binary',
	'sex_woman[T.Woman]',
	'ba_binary:sex_woman[T.Woman]',
	'Intercept'
])
sg.rename_covariates({
	'age':                          'Age',
	'I(age ** 2)':                  'Age-squared',
	'ba_binary':                    'BA degree',
	'sex_woman[T.Woman]':           'Gender (woman = 1)',
	'ba_binary:sex_woman[T.Woman]': 'BA × Woman',
	'Intercept':                    'Intercept'
})

sg

# Using the Linear Probability Model (LPM)

The LPM uses OLS on a binary dependent variable. It's quick and interpretable, though it can predict probabilities outside [0, 1].

Regress live music attendance (`binary_lvmus`) on BA education and log income.

In [ ]:
lpm_model1 = smf.ols(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(lpm_model1.summary())

# Using a Logit Model

Now, let's try logistic regression.

In [ ]:
logit_model1 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(logit_model1.summary())

Build a second logit model adding gender, age, and age-squared. Compare both.

In [ ]:
logit_model2 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont) + sex_woman + age + I(age**2)',
	data=df
).fit()
print(logit_model2.summary())

In [ ]:
# compare lpm and logit models using stargazer
sg = Stargazer([lpm_model1, logit_model1, logit_model2])
sg.title('Live Music Attendance: LPM vs Logit')
sg.dependent_variable_name('Attends live music')

sg.custom_columns(['LPM', 'Logit 1', 'Logit 2'], [1, 1, 1])
sg.covariate_order([
	'ba_binary',
	'np.log(income_cont)',
	'sex_woman[T.Woman]',
	'age',
	'I(age ** 2)',
	'Intercept'
])
sg.rename_covariates({
	'ba_binary':           'BA degree',
	'np.log(income_cont)': 'log(Income)',
	'sex_woman[T.Woman]':  'Gender (woman = 1)',
	'age':                 'Age',
	'I(age ** 2)':         'Age-squared',
	'Intercept':           'Intercept'
})

# The LPM coefficients are changes in probability, the logit coefficients are
# log-odds, so compare signs and significance across columns — not magnitudes.
sg



## Comparing Logit Models

### AIC comparison

In [ ]:
print(f'AIC logit_model1: {logit_model1.aic:.2f}')
print(f'AIC logit_model2: {logit_model2.aic:.2f}')

### Likelihood ratio test


In [ ]:
from scipy.stats import chi2

lr_stat = 2 * (logit_model2.llf - logit_model1.llf)
df_diff = logit_model2.df_model - logit_model1.df_model
p_value = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}')
print(f'df = {df_diff:.0f}')
print(f'p-value = {p_value:.4f}')

# Marginal Effects

The Python `marginaleffects` package mirrors R's package of the same name.

In [ ]:
# !pip install marginaleffects
from marginaleffects import avg_comparisons, comparisons

## Average marginal effect

The average marginal effect of a BA degree — equivalent to R's `avg_comparisons(logit_model1, variables='ba_binary')`.

In [ ]:
avg_comparisons(logit_model1, variables='ba_binary')

Interpretation: on average across our sample, having a BA degree increases the **probability** of attending a live music concert by X percentage points, holding other variables constant.

## Marginal effect at the mean

The marginal effect for a hypothetical person at the average of all X variables.

In [ ]:
comparisons(logit_model1, variables='ba_binary', newdata='mean')